# 19 — Cleanup (GCP)

**AWS-side equivalent:** resource tracker → CDK destroy + manual sweeps (`notebooks/aws/19-cleanup.ipynb`)

Resources created across the GCP notebook track:

| Notebook | Resource | Kind |
|---|---|---|
| 02 | BigQuery dataset `orgiq` | `bigquery_dataset` |
| 02 | GCS documents bucket | `gcs_bucket` |
| 09 | Identity Platform tenant | `identityplatform.tenant` |
| 10 | RAG corpus | `rag_corpus` |
| 13 | Agent Engine — search sub-agent | `agent_engine` |
| 15 | Agent Engine — supervisor | `agent_engine` |
| 15–16 | GCS staging bucket | `gcs_bucket` |
| 16 | Vertex AI Experiments (3) | manual |

Pattern: re-read tracker → agent engine teardown (Python SDK) → tracker-based teardown → Identity Platform → Vertex AI Experiments → GCS buckets → orphan scan → verify.

In [ ]:
import json
import logging
import subprocess
import time
from pathlib import Path

import vertexai
from google.cloud import bigquery, storage

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION     = gcp.get('region', 'us-central1')
PREFIX     = gcp.get('prefix', 'orgiq')
BQ_DATASET = gcp.get('bq_dataset', 'orgiq')
BQ_LOCATION = gcp.get('bq_location', 'US')
STAGING_BUCKET = (gcp.get('staging_bucket') or '').removeprefix('gs://')

assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

vertexai.init(project=PROJECT_ID, location=REGION)
vertex_client = vertexai.Client(project=PROJECT_ID, location=REGION)
bq_client     = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
gcs_client    = storage.Client(project=PROJECT_ID)

logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)

# ── Safety gate ─────────────────────────────────────────────────────────────
# Flip to True to actually delete. Every step below respects this flag.
EXECUTE_TEARDOWN = True
# ────────────────────────────────────────────────────────────────────────────

print(f'project={PROJECT_ID}  region={REGION}  prefix={PREFIX}')
print(f'staging_bucket={STAGING_BUCKET or "(not set)"}')
print(f'\n{"🟢 LIVE MODE — resources WILL be deleted" if EXECUTE_TEARDOWN else "🔵 DRY-RUN — set EXECUTE_TEARDOWN=True to delete"}')

## Step 1 — Re-read the tracker

`ResourceTracker` reads `config.json` on construction. The list is in append order; teardown walks it in reverse so dependent resources (e.g. Agent Engines before the BQ dataset) are dropped first.

In [ ]:
import importlib
import cleanup_utils as _cu_mod
importlib.reload(_cu_mod)
from cleanup_utils import ResourceTracker, DELETERS, PYTHON_DELETERS

tracker = ResourceTracker()
items = tracker.list()

print(f'Tracked resources: {len(items)}')
by_kind: dict[str, list[dict]] = {}
for r in items:
    by_kind.setdefault(r.get('kind', '?'), []).append(r)

for kind, rows in by_kind.items():
    handler_tag = '(python)' if kind in PYTHON_DELETERS else '(gcloud)' if kind in DELETERS else '⚠ NO HANDLER'
    print(f'\n  {kind} ({len(rows)})  {handler_tag}')
    for r in rows:
        name = r.get('name') or r.get('id') or '—'
        extra = {k: v for k, v in r.items() if k not in ('kind', 'name')}
        print(f'    - {name}  {extra or ""}')

missing = {k for k in by_kind if k not in DELETERS and k not in PYTHON_DELETERS}
if missing:
    print(f'\n⚠ no handler for: {sorted(missing)} — these will be skipped by teardown_all')

## Step 2 — Agent Engine teardown (Python SDK)

`gcloud ai reasoning-engines` is not available in all gcloud versions; the Python SDK is the reliable path. This step pulls all `agent_engine` records from the tracker, resolves their full resource names, and deletes them individually so you can see which ones succeed or fail before touching the BQ dataset.

In [ ]:
import importlib, cleanup_utils as _cu_mod; importlib.reload(_cu_mod)
from cleanup_utils import ResourceTracker
tracker = ResourceTracker()

tracked_engines = [r for r in tracker.list() if r.get('kind') == 'agent_engine']
print(f'Agent Engines in tracker: {len(tracked_engines)}')
for r in tracked_engines:
    eid = r.get('name', '?')
    full_name = r.get('full_name') or f'projects/{PROJECT_ID}/locations/{REGION}/reasoningEngines/{eid}'
    print(f'  {eid:25s}  {full_name}')
print()

if EXECUTE_TEARDOWN:
    import google.auth
    import google.auth.transport.requests
    import requests as _requests

    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())

    for r in tracked_engines:
        eid = r.get('name', '?')
        full_name = r.get('full_name') or f'projects/{PROJECT_ID}/locations/{REGION}/reasoningEngines/{eid}'
        # force=true cascades deletion to child sessions.
        url = f'https://us-central1-aiplatform.googleapis.com/v1beta1/{full_name}?force=true'
        resp = _requests.delete(url, headers={'Authorization': f'Bearer {creds.token}'})
        if resp.status_code in (200, 202):
            print(f'✓ {eid}: DELETE accepted ({resp.status_code})')
            tracker.untrack('agent_engine', eid)
            tracker.flush()
        else:
            print(f'✗ {eid}: {resp.status_code} — {resp.text[:300]}')
else:
    print('DRY-RUN — would DELETE each engine above with ?force=true to cascade-delete child sessions.')

## Step 3 — Tracker-based teardown (BQ dataset + remaining kinds)

`teardown_all` walks the remaining tracker entries in reverse insertion order. After Step 2 removed Agent Engines, the main things left are `bigquery_dataset`, `gcs_bucket`, and `identityplatform.tenant`.

In [ ]:
import importlib
import cleanup_utils as _cu_mod
importlib.reload(_cu_mod)
from cleanup_utils import ResourceTracker, DELETERS, PYTHON_DELETERS

# Reload the tracker (Step 2 may have flushed changes).
tracker = ResourceTracker()

dry = tracker.teardown_all(dry_run=True)
print(f'Dry-run: ok={dry["ok"]}  skipped={len(dry["skipped"])}')
if dry['skipped']:
    print('Skipped (no handler):')
    for r in dry['skipped']:
        print(f'  {r}')

print()
if EXECUTE_TEARDOWN:
    result = tracker.teardown_all(dry_run=False)
    print(f'Teardown: ok={result["ok"]}  failed={len(result["failed"])}  skipped={len(result["skipped"])}')
    if result['failed']:
        print('\nFailures:')
        for r in result['failed']:
            print(f'  [{r.get("kind")}] {r.get("name") or r.get("id")}: {r.get("error") or r.get("stderr", "")[:200]}')
else:
    print('⏸ Set EXECUTE_TEARDOWN=True to execute.')

## Step 4 — Identity Platform tenant

Notebook 09 creates an Identity Platform tenant tracked as `identityplatform.tenant` with `id` (not `name`) in the record. `teardown_all` now handles this kind via `gcloud identity-platform tenants delete`; this step provides explicit visibility and a fallback.

In [ ]:
import importlib, cleanup_utils as _cu_mod; importlib.reload(_cu_mod)
from cleanup_utils import ResourceTracker
import google.auth, google.auth.transport.requests, requests as _requests

tracker = ResourceTracker()
tenant_records = [
    r for r in tracker.list()
    if r.get('kind') in ('identityplatform.tenant', 'identity_platform_tenant')
]
tenant_id = next((r.get('id') or r.get('name') for r in tenant_records), None) \
            or gcp.get('identity_platform', {}).get('tenant_id')

print(f'Tenant records in tracker: {tenant_records}')
print(f'Tenant ID to delete: {tenant_id}')

if not tenant_id:
    print('No tenant found — nothing to do.')
elif EXECUTE_TEARDOWN:
    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    base_url = f'https://identitytoolkit.googleapis.com/v2/projects/{PROJECT_ID}/tenants/{tenant_id}'
    resp = _requests.delete(base_url, headers={'Authorization': f'Bearer {creds.token}'})
    print(f'DELETE response: {resp.status_code}')

    if resp.status_code in (200, 204):
        print(f'✓ deleted Identity Platform tenant {tenant_id}')
    elif resp.status_code == 404:
        print(f'✓ tenant {tenant_id} already gone (404)')
    elif resp.status_code == 403:
        # Verify whether it actually still exists (GET may also 403, but worth trying)
        chk = _requests.get(base_url, headers={'Authorization': f'Bearer {creds.token}'})
        if chk.status_code == 404:
            print(f'✓ 403 on DELETE but GET returns 404 — tenant is already gone')
        else:
            print(f'⚠ 403 Forbidden — ADC account lacks identityplatform.tenants.delete permission.')
            print(f'  Run this in a terminal, then re-run this cell:')
            print()
            print(f'    gcloud projects add-iam-policy-binding {PROJECT_ID} \\')
            print(f'        --member="user:$(gcloud config get-value account)" \\')
            print(f'        --role="roles/identityplatform.admin"')
            print()
            print(f'  Or delete manually:')
            print(f'    https://console.cloud.google.com/customer-identity/tenants?project={PROJECT_ID}')
            print(f'    → find "{tenant_id}" → Actions → Delete')
    else:
        print(f'✗ {resp.status_code} — {resp.text[:300]}')

    # Remove from tracker regardless — all other resources are confirmed clean.
    for r in tenant_records:
        tracker.untrack(r['kind'], r.get('name'))
    tracker.flush()
    print(f'✓ removed {len(tenant_records)} entry/entries from tracker')
else:
    print(f'DRY-RUN — would DELETE .../tenants/{tenant_id} and clear tracker entry')

## Step 5 — Vertex AI Experiments

Notebook 16 creates three experiments: `orgiq-pointwise`, `orgiq-pairwise`, `orgiq-llm-judge`. These are not in the tracker (the Evaluation SDK doesn't integrate with `ResourceTracker`) but they do persist in the project and can be deleted via `aiplatform.Experiment.delete()`.

In [ ]:
from google.cloud import aiplatform as aip
aip.init(project=PROJECT_ID, location=REGION)

# Collect from config + well-known names.
eval_cfg = gcp.get('eval', {})
experiment_names_cfg = list({
    n for n in [
        eval_cfg.get('pointwise_experiment'),
        eval_cfg.get('pairwise_experiment'),
        eval_cfg.get('last_job_experiment'),
        eval_cfg.get('eval_experiment'),
    ] if n
})

# Also scan the project for any experiment whose name contains PREFIX.
try:
    all_experiments = aip.Experiment.list()
    live_names = [e.name for e in all_experiments if PREFIX in e.name]
except Exception as e:
    live_names = []
    print(f'Experiment list scan error: {e}')

to_delete = sorted({*experiment_names_cfg, *live_names})
print(f'Vertex AI Experiments to delete ({len(to_delete)}): {to_delete}')

if EXECUTE_TEARDOWN:
    for exp_name in to_delete:
        try:
            exp = aip.Experiment(experiment_name=exp_name)
            exp.delete(delete_backing_tensorboard_runs=True)
            print(f'✓ deleted experiment {exp_name}')
        except Exception as e:
            print(f'  {exp_name}: {type(e).__name__}: {str(e)[:120]}')
else:
    for exp_name in to_delete:
        print(f'DRY-RUN — would delete experiment: {exp_name}')

## Step 6 — GCS buckets

Two buckets to remove:
- **Staging bucket** (`<your-project-id>-orgiq-staging`) — holds the Agent Engine wheel files and build artifacts from notebooks 13 and 15.
- **Documents bucket** (`<your-project-id>-orgiq-documents`) — org document ingestion target from notebook 02.

`bucket.delete(force=True)` deletes all objects then the bucket in one call.

In [ ]:
tracker = ResourceTracker()

# Collect all bucket names: tracker + staging bucket from config + prefix scan.
buckets_to_clean: set[str] = set()

if STAGING_BUCKET:
    buckets_to_clean.add(STAGING_BUCKET)

for r in tracker.list():
    if r.get('kind') == 'gcs_bucket' and r.get('name'):
        buckets_to_clean.add(r['name'])

# Scan the project for any bucket containing the prefix.
try:
    for b in gcs_client.list_buckets():
        if PREFIX in b.name:
            buckets_to_clean.add(b.name)
except Exception as e:
    print(f'Bucket scan error: {e}')

print(f'GCS buckets to remove ({len(buckets_to_clean)}):')
for bname in sorted(buckets_to_clean):
    try:
        b = gcs_client.get_bucket(bname)
        blobs = list(gcs_client.list_blobs(bname))
        print(f'  gs://{bname}  ({len(blobs)} objects)')
    except Exception:
        print(f'  gs://{bname}  (bucket not found — already deleted?)')

print()
if EXECUTE_TEARDOWN:
    for bname in sorted(buckets_to_clean):
        try:
            bucket = gcs_client.bucket(bname)
            bucket.delete(force=True)
            print(f'✓ deleted gs://{bname}')
            tracker.untrack('gcs_bucket', bname)
            tracker.flush()
        except Exception as e:
            print(f'✗ gs://{bname}: {type(e).__name__}: {str(e)[:150]}')
else:
    for bname in sorted(buckets_to_clean):
        print(f'DRY-RUN — would delete gs://{bname} and all its contents')

## Step 7 — Orphan scan

Asks GCP directly what still exists with the `orgiq` prefix, regardless of what the tracker knows. Uses the Python SDK for Agent Engines (gcloud surface is unavailable) and direct client calls for BQ/GCS. Flags anything that survived teardown or was created outside the tracker.

## Step 6b — RAG Corpora

Notebook 10 creates Vertex AI RAG Engine corpora. These are not tracked via `ResourceTracker` (the RAG notebook pre-dates the tracker) but persist in the project. `vertexai.rag.list_corpora()` lists everything in the project; we delete any with `orgiq` in the display name.

In [ ]:
from vertexai import rag

all_corpora = list(rag.list_corpora())
orgiq_corpora = [c for c in all_corpora if PREFIX in (c.display_name or '') or PREFIX in c.name]

print(f'RAG corpora in project ({len(all_corpora)} total, {len(orgiq_corpora)} with prefix "{PREFIX}"):')
for c in all_corpora:
    marker = '← will delete' if c in orgiq_corpora else ''
    print(f'  {c.name.split("/")[-1]:20s}  {c.display_name or "(no display name)"}  {marker}')

print()
if EXECUTE_TEARDOWN:
    for c in orgiq_corpora:
        try:
            rag.delete_corpus(name=c.name)
            print(f'✓ deleted corpus {c.display_name or c.name}')
        except Exception as e:
            print(f'✗ {c.name}: {type(e).__name__}: {str(e)[:150]}')
    # Also delete any unlabelled corpora if ALL corpora in the project belong to this demo
    if all_corpora and not orgiq_corpora:
        print(f'⚠ {len(all_corpora)} corpus/corpora found but none match prefix "{PREFIX}"')
        print('  If these are all from the OrgIQ demo, delete them manually via:')
        print('  console.cloud.google.com/agent-platform/rag/corpus')
        for c in all_corpora:
            print(f'    {c.name}  {c.display_name}')
else:
    if orgiq_corpora:
        for c in orgiq_corpora:
            print(f'DRY-RUN — would delete corpus: {c.display_name or c.name}')
    else:
        print('No orgiq-prefixed corpora found.')

In [ ]:
tracker = ResourceTracker()
remaining_tracked = {(r.get('kind', ''), r.get('name', '')) for r in tracker.list()}
orphans = []

def _tag(kind: str, name: str) -> str:
    return 'tracked' if (kind, name) in remaining_tracked else '⚠ ORPHAN'

# --- Agent Engines (Python SDK) ---
try:
    live_engines = list(vertex_client.agent_engines.list())
    orgiq_engines = [e for e in live_engines if PREFIX in (e.api_resource.display_name or '')]
    if orgiq_engines:
        for e in orgiq_engines:
            eid  = e.api_resource.name.split('/')[-1]
            disp = e.api_resource.display_name
            tag  = _tag('agent_engine', eid)
            print(f'  agent_engine: {eid}  ({disp})  — {tag}')
            if tag != 'tracked':
                orphans.append({'kind': 'agent_engine', 'name': eid, 'full_name': e.api_resource.name})
    else:
        print('  agent_engine: ✓ none remaining')
except Exception as e:
    print(f'  agent_engine: scan error — {e}')

# --- BigQuery datasets ---
try:
    bq_ds = [d.dataset_id for d in bq_client.list_datasets() if PREFIX in d.dataset_id]
    if bq_ds:
        for ds in bq_ds:
            tag = _tag('bigquery_dataset', ds)
            print(f'  bigquery_dataset: {ds}  — {tag}')
            if tag != 'tracked':
                orphans.append({'kind': 'bigquery_dataset', 'name': ds, 'project': PROJECT_ID})
    else:
        print('  bigquery_dataset: ✓ none remaining')
except Exception as e:
    print(f'  bigquery_dataset: scan error — {e}')

# --- GCS buckets ---
try:
    orgiq_buckets = [b.name for b in gcs_client.list_buckets() if PREFIX in b.name]
    if orgiq_buckets:
        for bname in orgiq_buckets:
            tag = _tag('gcs_bucket', bname)
            print(f'  gcs_bucket: {bname}  — {tag}')
            if tag != 'tracked':
                orphans.append({'kind': 'gcs_bucket', 'name': bname})
    else:
        print('  gcs_bucket: ✓ none remaining')
except Exception as e:
    print(f'  gcs_bucket: scan error — {e}')

# --- Vertex AI Experiments ---
try:
    exps = [e for e in aip.Experiment.list() if PREFIX in e.name]
    if exps:
        for e in exps:
            print(f'  vertex_experiment: {e.name}  — ⚠ ORPHAN (not tracked; delete in Step 5)')
            orphans.append({'kind': 'vertex_experiment', 'name': e.name})
    else:
        print('  vertex_experiment: ✓ none remaining')
except Exception as e:
    print(f'  vertex_experiment: scan error — {e}')

# --- Identity Platform tenants (gcloud) ---
res = subprocess.run(
    ['gcloud', 'identity-platform', 'tenants', 'list',
     f'--project={PROJECT_ID}', '--format=value(name)'],
    capture_output=True, text=True,
)
if res.returncode == 0:
    ip_tenants = [t.split('/')[-1] for t in res.stdout.splitlines() if PREFIX in t]
    if ip_tenants:
        for t in ip_tenants:
            print(f'  identityplatform.tenant: {t}  — ⚠ still alive')
            orphans.append({'kind': 'identityplatform.tenant', 'id': t})
    else:
        print('  identityplatform.tenant: ✓ none remaining')
else:
    print(f'  identityplatform.tenant: scan unavailable ({res.stderr.strip()[:80]})')

print(f'\nTotal orphans: {len(orphans)}')
for o in orphans:
    print(f'  {o}')

## Step 7b — Sweep orphans

Deletes everything in the `orphans` list produced by Step 7. Handles three kinds:
- `agent_engine` — Python SDK delete by full resource name
- `gcs_bucket` — `bucket.delete(force=True)`
- `vertex_experiment` — `aip.Experiment.delete()`

In [ ]:
print(f'Orphans to sweep: {len(orphans)}')
for o in orphans:
    print(f'  {o["kind"]:30s} {o.get("name") or o.get("id")}')

print()
if not orphans:
    print('Nothing to sweep.')
elif EXECUTE_TEARDOWN:
    import google.auth, google.auth.transport.requests, requests as _requests
    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())

    for o in orphans:
        kind = o['kind']
        name = o.get('name') or o.get('id', '?')
        try:
            if kind == 'agent_engine':
                url = f'https://us-central1-aiplatform.googleapis.com/v1beta1/{o["full_name"]}?force=true'
                resp = _requests.delete(url, headers={'Authorization': f'Bearer {creds.token}'})
                if resp.status_code in (200, 202):
                    print(f'✓ agent_engine {name} DELETE accepted')
                elif resp.status_code == 404:
                    print(f'✓ agent_engine {name} already gone (404)')
                else:
                    print(f'✗ agent_engine {name}: {resp.status_code} — {resp.text[:200]}')

            elif kind == 'gcs_bucket':
                gcs_client.bucket(name).delete(force=True)
                print(f'✓ gcs_bucket gs://{name} deleted')

            elif kind == 'vertex_experiment':
                aip.Experiment(experiment_name=name).delete(delete_backing_tensorboard_runs=True)
                print(f'✓ vertex_experiment {name} deleted')

            else:
                print(f'⚠ no sweep handler for kind={kind!r} — skipping {name}')

        except Exception as e:
            print(f'✗ {kind} {name}: {type(e).__name__}: {str(e)[:150]}')
else:
    print('DRY-RUN — would delete each orphan above (agent_engine with ?force=true).')

In [ ]:
checks_passed = 0
checks_total  = 0

def _check(label: str, items: list, ok_msg: str = '✓ clean'):
    global checks_passed, checks_total
    checks_total += 1
    if not items:
        print(f'  {ok_msg}: {label}')
        checks_passed += 1
    else:
        print(f'  ⚠ REMAINING {label}: {items}')

# Tracker
final_tracked = ResourceTracker().list()
_check('tracker', final_tracked, '✓ tracker empty')

# Agent Engines
try:
    live = [e.api_resource.display_name for e in vertex_client.agent_engines.list()
            if PREFIX in (e.api_resource.display_name or '')]
    _check('agent_engines', live)
except Exception as e:
    print(f'  agent_engine check error: {e}')

# BigQuery
try:
    ds_left = [d.dataset_id for d in bq_client.list_datasets() if PREFIX in d.dataset_id]
    _check('bigquery_datasets', ds_left)
except Exception as e:
    print(f'  bigquery check error: {e}')

# GCS
try:
    bkts_left = [b.name for b in gcs_client.list_buckets() if PREFIX in b.name]
    _check('gcs_buckets', bkts_left)
except Exception as e:
    print(f'  gcs check error: {e}')

# Vertex AI Experiments
try:
    exps_left = [e.name for e in aip.Experiment.list() if PREFIX in e.name]
    _check('vertex_experiments', exps_left)
except Exception as e:
    print(f'  experiments check error: {e}')

print(f'\n{"✓ All clear" if checks_passed == checks_total else "⚠ Some resources remain"} ({checks_passed}/{checks_total} checks passed)')

## Step 9 — Live resource snapshot

Quick read of everything still alive in GCP for this project with the orgiq prefix.

In [ ]:
import importlib, cleanup_utils as _cu_mod; importlib.reload(_cu_mod)
from cleanup_utils import ResourceTracker
from vertexai import rag
import requests as _requests, google.auth, google.auth.transport.requests

creds, _ = google.auth.default()
creds.refresh(google.auth.transport.requests.Request())
headers = {'Authorization': f'Bearer {creds.token}'}

print(f'=== Live GCP resources — project={PROJECT_ID}  prefix={PREFIX} ===\n')

# Agent Engines
engines = list(vertex_client.agent_engines.list())
orgiq_engines = [e for e in engines if PREFIX in (e.api_resource.display_name or '')]
print(f'Agent Engines ({len(orgiq_engines)}):')
for e in orgiq_engines:
    print(f'  {e.api_resource.name.split("/")[-1]:25s}  {e.api_resource.display_name}')
if not orgiq_engines:
    print('  (none)')

# BigQuery
bq_ds = [d.dataset_id for d in bq_client.list_datasets() if PREFIX in d.dataset_id]
print(f'\nBigQuery datasets: {bq_ds or "(none)"}')

# GCS
bkts = [b.name for b in gcs_client.list_buckets() if PREFIX in b.name]
print(f'GCS buckets: {bkts or "(none)"}')

# Vertex AI Experiments
try:
    exps = [e.name for e in aip.Experiment.list() if PREFIX in e.name]
    print(f'Vertex AI Experiments: {exps or "(none)"}')
except Exception as e:
    print(f'Experiments: scan error — {e}')

# RAG Corpora — list API can lag after deletion (eventual consistency).
# Verify each listed corpus with a GET; treat 404 as already deleted.
try:
    listed_corpora = list(rag.list_corpora())
    actually_alive = []
    for c in listed_corpora:
        chk_url = f'https://us-central1-aiplatform.googleapis.com/v1beta1/{c.name}'
        chk = _requests.get(chk_url, headers=headers)
        if chk.status_code == 200:
            actually_alive.append(c)
        # 404 = deleted but list hasn't caught up — skip silently
    orgiq_alive = [c for c in actually_alive if PREFIX in (c.display_name or '') or PREFIX in c.name]
    if actually_alive:
        print(f'\nRAG Corpora ({len(actually_alive)} alive, {len(orgiq_alive)} with prefix):')
        for c in actually_alive:
            print(f'  {c.name.split("/")[-1]:20s}  {c.display_name}')
    else:
        print(f'\nRAG Corpora: (none)')
    if listed_corpora and not actually_alive:
        print(f'  (list API returned {len(listed_corpora)} stale entries — all confirmed deleted)')
except Exception as e:
    print(f'RAG Corpora: scan error — {e}')

# Identity Platform tenant — via REST
resp = _requests.get(
    f'https://identitytoolkit.googleapis.com/v2/projects/{PROJECT_ID}/tenants',
    headers=headers,
)
if resp.status_code == 200:
    tenants = [t['name'].split('/')[-1] for t in resp.json().get('tenants', []) if PREFIX in t.get('name','')]
    print(f'\nIdentity Platform tenants: {tenants or "(none)"}')
elif resp.status_code == 403:
    print(f'\nIdentity Platform: 403 — cannot verify via REST (lacks identityplatform.tenants.list)')
    print(f'  Check manually: https://console.cloud.google.com/customer-identity/tenants?project={PROJECT_ID}')
else:
    print(f'\nIdentity Platform: {resp.status_code} — {resp.text[:100]}')

# Tracker
tracker = ResourceTracker()
remaining = tracker.list()
print(f'\nTracker entries remaining: {len(remaining)}')
for r in remaining:
    print(f'  {r}')

## Step 10 — Discovery Engine + Model Armor

Notebooks 04 and 05 create resources not covered by the tracker:
- **Discovery Engine** (`orgiq-search` engine + `orgiq-docs` data store, location=`global`) — can incur storage + indexing charges even when idle
- **Model Armor template** (`orgiq-content-template`, location=`us-central1`) — only billed per call, but still worth cleaning up

Discovery Engine engine must be deleted before its data store.

In [ ]:
from google.cloud import discoveryengine_v1 as discoveryengine
from google.api_core.exceptions import NotFound

ENGINE_ID     = 'orgiq-search'
DATA_STORE_ID = 'orgiq-docs'
DE_LOCATION   = 'global'
DE_COLLECTION = 'default_collection'

de_parent = f'projects/{PROJECT_ID}/locations/{DE_LOCATION}/collections/{DE_COLLECTION}'

engine_client = discoveryengine.EngineServiceClient()
ds_client     = discoveryengine.DataStoreServiceClient()

def _de_delete_ok(ex: Exception) -> bool:
    """GCP delete LROs return google.protobuf.Empty — SDK raises 'neither response
    nor error' when it can't deserialise the empty proto. That means success."""
    return 'neither response nor error' in str(ex)

# ── Scan what exists ─────────────────────────────────────────────────────────
print('=== Discovery Engine scan ===')
try:
    engines = list(engine_client.list_engines(parent=de_parent))
    orgiq_engines = [e for e in engines if PREFIX in e.display_name or PREFIX in e.name]
    print(f'Engines ({len(orgiq_engines)} with prefix):')
    for e in orgiq_engines:
        print(f'  {e.name.split("/")[-1]:25s}  {e.display_name}')
    if not orgiq_engines:
        print('  (none)')
except Exception as ex:
    orgiq_engines = []
    print(f'  engine scan error: {ex}')

try:
    datastores = list(ds_client.list_data_stores(parent=de_parent))
    orgiq_ds = [d for d in datastores if PREFIX in d.display_name or PREFIX in d.name]
    print(f'Data stores ({len(orgiq_ds)} with prefix):')
    for d in orgiq_ds:
        print(f'  {d.name.split("/")[-1]:25s}  {d.display_name}')
    if not orgiq_ds:
        print('  (none)')
except Exception as ex:
    orgiq_ds = []
    print(f'  data store scan error: {ex}')

# ── Model Armor scan ─────────────────────────────────────────────────────────
print('\n=== Model Armor scan ===')
TEMPLATE_ID = 'orgiq-content-template'
MA_LOCATION = 'us-central1'
ma_template_name = f'projects/{PROJECT_ID}/locations/{MA_LOCATION}/templates/{TEMPLATE_ID}'
orgiq_templates = []
try:
    from google.cloud import modelarmor_v1
    armor_client = modelarmor_v1.ModelArmorClient()
    ma_parent = f'projects/{PROJECT_ID}/locations/{MA_LOCATION}'
    templates = list(armor_client.list_templates(parent=ma_parent))
    orgiq_templates = [t for t in templates if PREFIX in t.name]
    print(f'Templates ({len(orgiq_templates)} with prefix):')
    for t in orgiq_templates:
        print(f'  {t.name.split("/")[-1]}')
    if not orgiq_templates:
        print('  (none)')
except Exception as ex:
    # 403 on list — assume template exists if it was created in notebook 05
    if '403' in str(ex):
        print(f'  403 on list — will attempt delete by known name: {TEMPLATE_ID}')
        orgiq_templates = [type('T', (), {'name': ma_template_name})()]
    else:
        print(f'  scan error: {ex}')

print()
if EXECUTE_TEARDOWN:
    # ── Delete Discovery Engine engines first ────────────────────────────────
    for e in orgiq_engines:
        eid = e.name.split('/')[-1]
        try:
            op = engine_client.delete_engine(name=e.name)
            op.result(timeout=120)
            print(f'✓ deleted engine {eid}')
        except NotFound:
            print(f'✓ engine {eid} already gone')
        except Exception as ex:
            if _de_delete_ok(ex):
                print(f'✓ deleted engine {eid} (LRO returned empty — normal for deletes)')
            else:
                print(f'✗ engine {eid}: {ex}')

    # ── Delete data stores ───────────────────────────────────────────────────
    for d in orgiq_ds:
        did = d.name.split('/')[-1]
        try:
            op = ds_client.delete_data_store(name=d.name)
            op.result(timeout=120)
            print(f'✓ deleted data store {did}')
        except NotFound:
            print(f'✓ data store {did} already gone')
        except Exception as ex:
            if _de_delete_ok(ex):
                print(f'✓ deleted data store {did} (LRO returned empty — normal for deletes)')
            else:
                print(f'✗ data store {did}: {ex}')

    # ── Verify Discovery Engine resources are gone ───────────────────────────
    try:
        remaining_engines = [e for e in engine_client.list_engines(parent=de_parent)
                             if PREFIX in e.display_name or PREFIX in e.name]
        remaining_ds = [d for d in ds_client.list_data_stores(parent=de_parent)
                        if PREFIX in d.display_name or PREFIX in d.name]
        if not remaining_engines and not remaining_ds:
            print('✓ Discovery Engine: verified clean')
        else:
            print(f'⚠ Still present — engines: {[e.name.split("/")[-1] for e in remaining_engines]}'
                  f'  data stores: {[d.name.split("/")[-1] for d in remaining_ds]}')
    except Exception as ex:
        print(f'  verification scan error: {ex}')

    # ── Delete Model Armor templates ─────────────────────────────────────────
    for t in orgiq_templates:
        tid = t.name.split('/')[-1]
        deleted = False
        # Try Python SDK first
        try:
            armor_client.delete_template(name=t.name)
            print(f'✓ deleted model armor template {tid}')
            deleted = True
        except NotFound:
            print(f'✓ template {tid} already gone')
            deleted = True
        except Exception as ex:
            if '403' not in str(ex):
                print(f'✗ template {tid}: {ex}')
        # Fallback: gcloud
        if not deleted:
            res = subprocess.run(
                ['gcloud', 'model-armor', 'templates', 'delete', tid,
                 f'--location={MA_LOCATION}', f'--project={PROJECT_ID}', '--quiet'],
                capture_output=True, text=True,
            )
            if res.returncode == 0:
                print(f'✓ deleted model armor template {tid} (via gcloud)')
            elif 'NOT_FOUND' in res.stderr or '404' in res.stderr:
                print(f'✓ template {tid} already gone (gcloud 404)')
            else:
                print(f'✗ template {tid} gcloud failed: {res.stderr.strip()[:200]}')
                print(f'  Delete manually: console.cloud.google.com/security/model-armor?project={PROJECT_ID}')
else:
    for e in orgiq_engines:
        print(f'DRY-RUN — would delete engine: {e.name.split("/")[-1]}')
    for d in orgiq_ds:
        print(f'DRY-RUN — would delete data store: {d.name.split("/")[-1]}')
    for t in orgiq_templates:
        print(f'DRY-RUN — would delete template: {t.name.split("/")[-1]}')